# 09 — Terraform, Runtime e Identity M2M

Referência: [AgentCore workshops](https://github.com/awslabs/agentcore-samples/tree/main/06-workshops). O código e o IaC desta jornada adaptam os conceitos às contas do workshop.

## Pré-requisito: 1 conta AWS

Esta aula não exige fronteiras entre contas. Seus stacks, integrações e permissões precisam estar implantados na conta da plataforma. Ela não funciona automaticamente em uma conta AWS vazia.

**Runner CodeBuild do projeto:** a seleção é declarada no deploy. A jornada `single-account` inclui 00–09 e 15–16; Registry, Terraform e RAG são exigidos somente pelas aulas que os utilizam. Ampliar a seleção exige atualizar a implantação pelo IaC. Confira a [matriz de contas e execução](../docs/contas-e-elegibilidade.md).

![Arquitetura AWS da aula 09](../docs/diagramas/svg/09_terraform_runtime_gateway_identity.svg)

Leia as fronteiras do diagrama e confronte-as com o IaC e as chamadas abaixo.

Todos os serviços e agentes desta jornada são implantados na AWS.
O notebook é o roteiro e cliente das APIs; ferramentas, inferência,
políticas, fontes e memória executam nos serviços indicados.
Use o ambiente AWS descrito no [guia de implantação](../docs/deploy.md).
Configure os identificadores dos stacks antes de executar. Uma
dependência ausente bloqueia a aula antes das chamadas do exercício.
Antes de usar “Run All”, selecione as aulas compatíveis com as contas
preparadas. O bloco 10–14 exige quatro contas na implantação atual.

### Carregar a configuração e conferir as dependências

A configuração contém perfis/roles e IDs, nunca senhas ou tokens. O STS confirma a conta antes de consultar os stacks; o manifesto exige as dependências desta aula.

In [ ]:
import json
import os
import uuid
from pathlib import Path
from IPython.display import Code, display
import inspect
from santander_demo.aws_workshop.client import AWSWorkshop
from santander_demo.aws_workshop.config import load
from santander_demo.aws_workshop.manifest import require

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CONFIG = Path(os.environ.get("WORKSHOP_CONFIG", ROOT / "config/aws.workshop.json"))
config = load(CONFIG)
dependencies = require(config, "09")
print({"lab": "09", "dependencies": list(dependencies), "region": config["region"]})

### Ler os recursos Terraform do caminho M2M

O HCL declara Cognito, Identity, Runtime, Gateway e Lambda. O Runtime usa sua role para obter uma credencial de aplicação pelo Identity. M2M não representa permissões de uma pessoa.

In [ ]:
for filename in ["main.tf", "identity.tf", "iam.tf"]:
    display(Code((ROOT / "infra/terraform_agentcore" / filename).read_text(), language="terraform"))

### Carregar os outputs validados da implantação

O pipeline Terraform verifica a conta, gera um plano revisável e só aplica esse plano quando os hashes dos arquivos e artefatos continuam iguais. Os outputs abaixo contêm identificadores, sem client_secret ou tokens.

In [ ]:
tf = json.loads((ROOT / dependencies["terraform"]["outputs"]).read_text())
settings = json.loads((ROOT / dependencies["terraform"]["config"]).read_text())
import boto3
aws = boto3.Session(profile_name=settings.get("profile"), region_name=settings["region"])
assert aws.client("sts").get_caller_identity()["Account"] == tf["account_id"]
print({k: tf[k] for k in ["runtime_arn", "gateway_url", "workload_name", "provider_name"]})

### Executar Identity → Gateway → Lambda dentro do Runtime

Duas chamadas verificam clientes sintéticos diferentes. A terceira inclui inferência Bedrock no Runtime. O verificador exige IDs de requisição do Gateway e da Lambda.

In [ ]:
from santander_demo.terraform_lab.client import invoke_runtime
display(Code(inspect.getsource(invoke_runtime), language="python"))
for customer in ["C001", "C002"]:
    print(invoke_runtime(tf, aws, customer_id=customer))
print(invoke_runtime(tf, aws, mode="llm"))

**Equivalência com Entra:** um cliente confidencial Entra também pode obter um token de aplicação. Client credentials continua sendo M2M; delegar a pessoa exige um fluxo e permissões diferentes.

### Evidências e encerramento

Registre IDs de requisição e decisões, sem tokens ou senhas. Os recursos são compartilhados entre aulas: a remoção pertence ao pipeline de limpeza do projeto e deve respeitar essa dependência.